# 💳 Credit Risk Model — German Credit Data

**Objectif :** prédire si une demande de crédit est à risque faible (`good`) ou élevé (`bad`).

**Étapes :**
1. Chargement et nettoyage des données
2. Analyse exploratoire (EDA)
3. Feature engineering (encodage)
4. Entraînement et comparaison de modèles à base d'arbres
5. Export du meilleur modèle pour l'app Streamlit

## 1. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

pd.set_option("display.max_columns", None)  # affiche toutes les colonnes
sns.set_style("whitegrid")                  # style des graphiques

## 2. Chargement des données

In [ ]:
df = pd.read_csv("data/german_credit_data.csv")
df = df.drop(columns=["Unnamed: 0"])  # simple index, inutile
df.head()

## 3. Exploration et nettoyage

In [ ]:
print("Taille :", df.shape)
print("\nValeurs manquantes :\n", df.isna().sum())
print("\nDoublons :", df.duplicated().sum())
print("\nCible :\n", df["Risk"].value_counts())

In [ ]:
# Résumé statistique (include="all" ajoute les colonnes texte)
df.describe(include="all").T

Les colonnes `Saving accounts` et `Checking account` ont des valeurs manquantes.
Choix : **supprimer ces lignes** plutôt que d'inventer une valeur, car l'état des comptes
est une info clé pour évaluer un crédit. On passe de 1000 à ~520 lignes.

In [ ]:
df = df.dropna().reset_index(drop=True)
print("Taille après nettoyage :", df.shape)

## 4. Analyse exploratoire
### 4.1 Variables numériques

In [ ]:
num_cols = ["Age", "Credit amount", "Duration"]

# Histogrammes : forme de la distribution
df[num_cols].hist(bins=15, edgecolor="black", figsize=(12, 4))
plt.suptitle("Distribution des variables numériques", fontsize=14)
plt.show()

In [ ]:
# Boxplots : repérer les valeurs extrêmes (outliers)
plt.figure(figsize=(14, 5))
for i, col in enumerate(num_cols):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(y=df[col], color="skyblue")
    plt.title(col)
plt.tight_layout()
plt.show()

### 4.2 Variables catégorielles

In [ ]:
cat_cols_plot = ["Sex", "Job", "Housing", "Saving accounts", "Checking account", "Purpose"]

plt.figure(figsize=(14, 10))
for i, col in enumerate(cat_cols_plot):
    plt.subplot(2, 3, i + 1)
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)  # trié par fréquence
    plt.title(col)
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.3 Corrélations

In [ ]:
corr = df[["Age", "Job", "Credit amount", "Duration"]].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matrice de corrélation")
plt.show()

Seule corrélation notable : **montant ↔ durée** (~0.6) — les gros crédits sont remboursés sur plus longtemps.

### 4.4 Lien avec le risque

In [ ]:
# Répartition de la cible en %
print((df["Risk"].value_counts(normalize=True) * 100).round(1))

plt.figure(figsize=(14, 4))
for i, col in enumerate(num_cols):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(data=df, x="Risk", y=col)
    plt.title(f"{col} par risque")
plt.tight_layout()
plt.show()

# Moyennes par classe de risque
df.groupby("Risk")[num_cols].mean()

In [ ]:
plt.figure(figsize=(14, 10))
for i, col in enumerate(cat_cols_plot):
    plt.subplot(2, 3, i + 1)
    sns.countplot(data=df, x=col, hue="Risk", order=df[col].value_counts().index)
    plt.title(f"{col} par risque")
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**À retenir :**
- Les crédits `bad` ont en moyenne un **montant plus élevé** et une **durée plus longue**.
- Plus le compte courant est fourni, plus le risque baisse.
- L'âge semble peu lié au risque.

## 5. Feature engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

features = ["Age", "Sex", "Job", "Housing", "Saving accounts",
            "Checking account", "Credit amount", "Duration"]
target = "Risk"
df_model = df[features + [target]].copy()

# Encodage texte -> nombres. Chaque encodeur est sauvegardé pour l'app
# (l'app doit transformer les entrées exactement de la même façon).
cat_cols = ["Sex", "Housing", "Saving accounts", "Checking account"]
for col in cat_cols:
    enc = LabelEncoder()
    df_model[col] = enc.fit_transform(df_model[col])
    joblib.dump(enc, f"models/{col}_encoder.pickle")

# Encodage de la cible : bad = 0, good = 1 (ordre alphabétique)
target_encoder = LabelEncoder()
df_model[target] = target_encoder.fit_transform(df_model[target])
joblib.dump(target_encoder, "models/target_encoder.pickle")

df_model.head()

> Pas de normalisation (scaling) : les modèles à base d'arbres n'en ont pas besoin.

## 6. Séparation train / test

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=target)
y = df_model[target]

# 80 % entraînement / 20 % test
# stratify=y : garde la même proportion good/bad dans les deux jeux
# random_state=1 : résultats reproductibles
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1)

print("Train :", X_train.shape, "| Test :", X_test.shape)

## 7. Modélisation

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

def train_model(model, param_grid):
    """Teste toutes les combinaisons de param_grid (validation croisée 5 plis),
    garde la meilleure, puis l'évalue sur le jeu de test."""
    grid = GridSearchCV(model, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    acc = accuracy_score(y_test, best.predict(X_test))
    return best, acc, grid.best_params_

In [ ]:
# class_weight="balanced" : compense le déséquilibre good/bad
dt_grid = {"max_depth": [3, 5, 7, 10, None],
           "min_samples_split": [2, 5, 10],
           "min_samples_leaf": [1, 2, 4]}

# Même grille pour Random Forest et Extra Trees
forest_grid = {"n_estimators": [100, 200],
               "max_depth": [5, 7, 10, None],
               "min_samples_split": [2, 5, 10],
               "min_samples_leaf": [1, 2, 4]}

models = {
    "Decision Tree": (DecisionTreeClassifier(random_state=1, class_weight="balanced"), dt_grid),
    "Random Forest": (RandomForestClassifier(random_state=1, class_weight="balanced", n_jobs=-1), forest_grid),
    "Extra Trees":   (ExtraTreesClassifier(random_state=1, class_weight="balanced", n_jobs=-1), forest_grid),
}

results = {}
for name, (model, grid) in models.items():
    best, acc, params = train_model(model, grid)
    results[name] = {"model": best, "accuracy": acc, "params": params}
    print(f"{name:15} accuracy = {acc:.3f} | {params}")

## 8. Choix et export du meilleur modèle

In [ ]:
best_name = max(results, key=lambda k: results[k]["accuracy"])
best_model = results[best_name]["model"]
print(f"Meilleur modèle : {best_name} ({results[best_name]['accuracy']:.3f})")

joblib.dump(best_model, "models/credit_model.pickle")

In [ ]:
# Importance des variables : ce qui pèse le plus dans la décision du modèle
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()
importances.plot(kind="barh", figsize=(8, 4), title=f"Importance des variables — {best_name}")
plt.show()

## 9. Test rapide d'une prédiction

In [ ]:
encoders = {col: joblib.load(f"models/{col}_encoder.pickle") for col in cat_cols}

exemple = pd.DataFrame({
    "Age": [30],
    "Sex": encoders["Sex"].transform(["male"]),
    "Job": [1],
    "Housing": encoders["Housing"].transform(["own"]),
    "Saving accounts": encoders["Saving accounts"].transform(["little"]),
    "Checking account": encoders["Checking account"].transform(["moderate"]),
    "Credit amount": [1000],
    "Duration": [12],
})

pred = best_model.predict(exemple)[0]
print("Prédiction :", "good (risque faible)" if pred == 1 else "bad (risque élevé)")

## Conclusion
- Meilleure accuracy obtenue : **~0.67** (Random Forest / Extra Trees), contre 0.58 pour un arbre seul.
- Le modèle est exporté dans `models/` et utilisé par `app.py`.

**Pistes d'amélioration :** garder les lignes avec valeurs manquantes (catégorie "inconnu"),
utiliser `Purpose`, tester XGBoost, regarder d'autres métriques (recall, F1) car rater un client `bad` coûte cher.